# Making My Thoughts Bilingual with Agentic AI

Hi there! It's been a while :)

Let's begin today's post with the most obvious statement, which you probably already know if you are here: **I am Brazilian**. However, I have worked for multinational companies for the past five to seven years, so writing in English now comes naturally to me. I code in English, and [my master's thesis](https://teses.usp.br/teses/disponiveis/45/45134/tde-25092025-141609/publico/MScThesisAESAndreBarbosaReviewed.pdf) was written in English. My grammar is not always perfect, but I try my best to keep improving :)

Writing in English also increases my technical reach. These days, I have colleagues who are not Brazilian, and I would like them to be able to read about my thought process. But what about more junior Brazilian readers? When I entered university, my English skills were poor, and it would have been wonderful to have access to more technical posts in Brazilian Portuguese.

At the same time, not every idea has to begin in English. Sometimes I may want to write naturally in Portuguese while still giving my international colleagues an English counterpart. Ideally, I could write each post in whichever language feels right and make it available in the other one as well. This would be the best of both worlds!

However, this blog is a part-time hobby, and I simply do not have time to write every post twice. I could ask readers to translate the page themselves, but that would shift the effort to them. I could also pay for a translation API, but I wanted to spend as little as possible on this hobby.

Then a natural question popped into my head:

> Could I use today's coding agents to design inexpensive experiments, select a translation model, and build a practical translation workflow in both directions?

I am a trained data scientist, and I like languages and natural language processing (NLP), so this sounded like a fun idea to explore. Rather than training a model from scratch, I could use existing pretrained models, write each post once, and review the generated counterpart. That is the story I want to tell here :)

# Augmenting Human Capabilities, Not Replacing Them

This idea reminds me of a former manager. We once worked on a product for our company's customer-support team, and I often argued that we should build AI systems that augment people rather than replace them. We should also be responsible and transparent in how we think about and develop such products. He pointed me to [this article](https://hbr.org/2021/03/ai-should-augment-human-intelligence-not-replace-it), which shaped much of how I think about AI products.

How does that idea relate to this post? I do not want a process that translates content without oversight. I want an auditable process that is autonomous enough not to disrupt my development workflow. After all, I write this blog in my spare time and need to use that time carefully. How could I achieve this? GitHub Actions!

## Architecture at a Glance

At a high level, the pipeline will look like this:

![](images\scaling-translation-posts\architeture-automate-translation-v2.png)

*This diagram was created with the help of ChatGPT Sol using high reasoning effort.*


The intended workflow starts only after the source post has been reviewed and its PR has been merged. The resulting push to `main` triggers a GitHub Actions job that translates the post and opens a separate translation PR. The CI environment therefore needs a model that is inexpensive and can run on CPU. It also builds and publishes a temporary preview so I can review the translated page before merging the translation PR. When that PR is closed, the temporary preview is removed.

The challenge? Find a reliable model

# Searching for a Good Model

I could blindly use a self-hosted model from Hugging Face, but as a scientist, I need to understand how well it performs :) What is the best way to do that? Evaluations!

## Constructing a Reference Dataset

I did not want a literal, word-for-word translation; I wanted a translation that made sense. Fortunately, I had previously written the same post in both [English](https://github.com/abarbosa94/personal_blog/blob/e78386012f37512a5ebd316a1389fabf9bf3b707/_notebooks/2020-09-19-Distilling-BERT.ipynb) and [Portuguese](https://github.com/abarbosa94/personal_blog/blob/e78386012f37512a5ebd316a1389fabf9bf3b707/_notebooks/2020-09-19-Distilling-BERT-pt.ipynb). These notebooks gave me comparable bilingual source material. However, their sentences were not necessarily aligned: sentence 0 in the English post was not guaranteed to correspond to sentence 0 in the Portuguese post. How could I construct the pairs correctly?

To map and align English–Portuguese pairs, I used [LaBSE](https://huggingface.co/sentence-transformers/LaBSE) embeddings (**La**nguage-agnostic **B**ERT **S**entence **E**mbedding) [@feng2022labse]. For an English sentence $e$ and a Portuguese sentence $p$, I calculated:

$$
\operatorname{similarity}(e,p)
=\cos\!\left(\operatorname{LaBSE}(e),\operatorname{LaBSE}(p)\right)
$$

A high cosine similarity suggests that the passages express similar meanings even when they have little surface overlap.

For example:

> We use the CLS token representation.

and:

> Utilizamos a representação do token CLS.

have little surface overlap, but LaBSE should place them close together because their meanings correspond.



## Obtaining a Global Alignment

Codex GPT 5.6 Sol suggested this follow-up approach. I found it clever, so I also asked why it was necessary. Here is my paraphrase of the explanation:

Selecting the most similar Portuguese sentence independently for every English sentence could reuse sentences or scramble their order. LaBSE therefore supplied the **semantic similarity signal**, but I still needed the best **global alignment**. Dynamic programming provided a way to find it.


The alignment algorithm enforced:

- Monotonic order: later English content maps to later Portuguese content.
- No overlapping or reused passages.
- One-to-many and many-to-one alignments, up to three sentences per side.
- Gaps for content present in only one language.

::: {.callout-note collapse="true"}
Conceptually, each candidate alignment received:

$$
\text{alignment score}
=\text{LaBSE similarity}
-\text{merge penalty}
-\text{length-mismatch penalty}
$$

The algorithm then found the sequence of alignments with the greatest total score. Gaps received their own negative penalty.

This is implemented in the experiment's [evaluation pipeline](https://github.com/abarbosa94/personal_blog/blob/main/experiments/scaling-my-posts/src/translation_eval.py).
:::

## From Sentence Pairs to a Trusted Dataset

Before evaluating model outputs, I needed to validate the bilingual dataset itself. I asked Codex to create a review interface following guidance from the [*Evals for AI Engineers* book](https://www.oreilly.com/library/view/evals-for-ai/9798341660717/). A screenshot of the interface appears below:

![](images\scaling-translation-posts\ui-review-translate.jpg)


The idea is simple: given the candidate pairs proposed by LaBSE and dynamic programming, do I agree with each alignment?


The interface shows an English segment and its proposed Portuguese counterpart. I could override either side when a documented correction was necessary, then choose `Accept` when the segments were well aligned, `Localize` when one side intentionally paraphrased the other, `Exclude` when the pair was unsuitable, or `Defer` when I was uncertain. After this review, I had a trustworthy evaluation dataset containing [36 accepted pairs](https://github.com/abarbosa94/personal_blog/blob/main/posts/data/translation-eval-segments.csv)—enough for an initial minimum viable product (MVP).

## Selecting Translator Candidates

With the trusted dataset in place, I needed candidate translation models. After some initial research, I selected three:

- [Marian OPUS-MT](https://huggingface.co/Helsinki-NLP/opus-mt-tc-big-en-pt), which is inexpensive and lightweight. I also remembered experimenting with Marian while working on the [Bergamot project](https://browser.mt/).
- [NLLB-200 distilled 600M](https://huggingface.co/facebook/nllb-200-distilled-600M).
- [Tower+ 2B](https://huggingface.co/Unbabel/Tower-Plus-2B).



Because my goal is to run this workflow on free GitHub Actions runners, inference needs to work on a **CPU**. It can be slow, but the model must run without a GPU. This constraint is also why I selected the 2B-parameter Tower+ model even though larger versions are available.

::: {.callout-warning collapse="true" title="After some initial experiments, I Screened Out NLLB"}
I evaluated `facebook/nllb-200-distilled-600M` at revision `f8d333a098d19b4fd9a8b18f94170487ad3f821d`. It ran on CPU with Transformers 4.53.2, forced the target-language beginning-of-sentence token, used four-beam generation, truncated inputs at 512 tokens, and set the output limit to `min(512, max(32, 2 * longest_source_tokens + 32))`.

The original exclusion was exploratory rather than preregistered. To make that decision auditable, I defined a simple screening rule. Let's considerer the following example: if the normalized four-token sequence `in accordance with the` appeared at least **four times in one prediction**, that prediction was flagged as potentially repetitive. A model-direction failed when more than **5%** of its segments were flagged, and a candidate had to pass in both directions.

The rule flagged 0 of 36 English-to-Portuguese outputs and 4 of 36 Portuguese-to-English outputs (11.1%). For example, segment `p08-a01`, whose source was “Predição da Próxima Sentença (Next Sentence Prediction — NSP),” produced an unrelated sentence beginning “The Commission shall adopt delegated acts...” and repeated “in accordance with the opinion” **nine times**. The other flagged segments were `p05-a04`, `p06-a02`, and `p08-a03`. These IDs can be checked in the [reference dataset](https://github.com/abarbosa94/personal_blog/blob/main/posts/data/translation-eval-segments.csv).

Because I needed one model that worked in both directions, I excluded NLLB from the LLM-judge comparison while retaining it in the timing and overlap appendix. The [screening code](https://github.com/abarbosa94/personal_blog/blob/main/experiments/scaling-my-posts/src/screening.py) and saved predictions make this retrospective rule reproducible.
:::

## The Metrics

I wanted to apply techniques I learned from the [*Evals for AI Engineers* book](https://www.oreilly.com/library/view/evals-for-ai/9798341660717/). For this experiment, I created initial LLM judges...but what should they measure?

### MQM

With the help of ChatGPT, I found [MQM](https://themqm.org/), a framework for analytic Translation Quality Evaluation [@mqm-scoring]. In this experiment, I automated an MQM-style rubric with an LLM judge. In a nutshell, it tries to answer this question: _What kinds of translation errors occurred, and how serious were they?_


The judge compares the source, candidate, and human reference. It marks errors by category and severity. The defined categories include accuracy, omission, addition, fluency, terminology, locale, style, and formatting.

The formula is:

$$
P_s = W_{\text{minor}}N_{\text{minor}} + W_{\text{major}}N_{\text{major}} + W_{\text{critical}}N_{\text{critical}}
$$

Here, $N$ means the number of errors identified and $W$ means the _weight_ applied to the error category.

::: {.callout-note collapse="true"}
## Our MQM Implementation
The severety weights in my experiment were the following:

$$
\left\{
\begin{aligned}
W_\text{minor} &= 1 \\
W_\text{major} &= 5 \\
W_\text{critical} &= 10
\end{aligned}
\right.
$$

Example: a segment with two minor terminology problems and one major omission receives:

$$
2(1)+1(5)+0(0)=7
$$

These weights should not be interpreted as the universal definition of MQM scoring.
:::

In the final metric, zero means that the judge reported no errors so lower is better.

### Pairwise Preference

To calibrate the judges, I also needed to compare model outputs with human preferences. This leads to a simple question: _If two translations are placed side by side, which one is better overall?_

For each source segment, the judge chooses candidate A, candidate B, or a tie. We run the comparison twice, reversing the candidate order.

A comparison is stable only if reversing the display order produces the same underlying result. For example:

- Model $X$ as A versus Model $Y$ as B: B wins.
- Model $Y$ as A versus Model $X$ as B: A wins.

Both judgments mean that Model $Y$ won, so the comparison is stable. A stable tie contributes half a point to each model. The preference rate is therefore:


$$
R_X =
\frac{\text{wins by }X + 0.5\times\text{ties involving }X}
{\text{stable comparisons involving model }X}
$$

Higher is better. Unstable comparisons are reported separately because they indicate order sensitivity or judge uncertainty.

Both zero-shot judge prompts are available in the experiment's [prompt source file](https://github.com/abarbosa94/personal_blog/blob/main/experiments/scaling-my-posts/src/prompts.py). For this hobby-project experiment, I selected Kimi K3 as the judge and prepaid USD 20 in API credit (approximately BRL 100).

#### Human–Judge Agreement

Pairwise preference evaluation indicates whether the judge prefers model $X$ or model $Y$. However, it does not show whether the judge's decisions are consistent with my own. To assess this consistency, I computed Cohen's kappa [@kappaReview1]:

$$
\kappa = \frac{p_o - p_e}{1 - p_e}
$$

where (p_o) is the observed agreement and (p_e) is the agreement expected by chance based on the evaluators' label frequencies.

I also computed the raw agreement:

$$
A = \frac{\text{human–judge matches}}{\text{reviewed stable items}}
$$

Raw agreement is straightforward to interpret. Cohen's kappa complements it by accounting for agreement that may occur by chance (e.g., when both evaluators tend to select the same label frequently).


# Calibrating the Judge

As stated, MQM helped identify specific translation problems, while pairwise preference enabled me to measure how often the pairwise-judge agreed with my own choices.

Because this was an MVP, I deliberately sampled known disagreements, close calls, and clear cases. In short, I first wanted to stress-test the judge's behavior. This produced 18 review cases.

Of the 18 selected items:

- I reviewed 17 and deferred 1.
- Among my 17 completed items, 3 had unstable automated decisions.
- That left 14 completed items with stable judge answers.

Therefore, the reported agreement was:

$$
\frac{11\text{ agreements}}{14\text{ comparable items}}=78.6\%
$$

The observed agreement was 78.6%, exceeding my exploratory quality gate of 70% (which is essentially a magic number that I chose). However, this result was based on only 14 comparable items, so the estimate is imprecise: the 95% bootstrap interval ranged from 57.1% to 100%. The sample was also deliberately stress-oriented, with an emphasis on disagreements, borderline translations, and other difficult cases. Consequently, this interval should not be interpreted as the judge’s expected agreement rate on future posts. I treated the result as encouraging pilot evidence. This was enough evidence to continue the experiment, but not enough to treat the judge as conclusively validated.

I also computed Cohen's kappa, obtaining $\kappa=0.672$. The interpretation of this result can vary across domains and should therefore be treated as a guideline rather than a universal threshold. For example, in automatic essay scoring, $\kappa$ values between 0.4 and 0.75 are considered fair-to-good agreement [@kappaReview1; @kappaReview2].

# Benchmarking the Results

After the judge-calibration pilot, I proceeded with the MQM and pairwise evaluations. Following the bidirectional screening rule described above, I excluded NLLB from this paid stage and compared Marian OPUS-MT with Tower+ 2B in both English-to-Portuguese and Portuguese-to-English translation.

  | Direction | Model | Mean MQM penalty | Median MQM penalty | Pairwise preference |
  |---|---|---:|---:|---:|
  | EN to PT-BR | Marian OPUS-MT | 3.67 | 3.0 | 20.6% |
  | EN to PT-BR | Tower+ 2B | 1.92 | **1.0** | **79.4%** |
  | PT-BR to EN | Marian OPUS-MT | 2.56 | 1.0 | 21.0% |
  | PT-BR to EN | Tower+ 2B | 0.69 | **0.0** | **79.0%** |

For MQM, lower is better. For pairwise preference, higher is better.

Tower+ produced fewer and less severe errors in both directions (according to the zero-shot Kimi K3 judge):

- English to Portuguese: the mean penalty fell from 3.67 to 1.92, a 48% reduction.
- Portuguese to English: the mean penalty fell from 2.56 to 0.69, a 73% reduction.

The median penalty for Tower+ in Portuguese-to-English translation was zero. This means at least half of its translations received no MQM penalty from the judge.

## MQM Error Analysis

Neither model received a critical error. Tower+ nevertheless reduced both minor and major errors substantially, with its strongest result in Portuguese-to-English translation.

  | Direction | Model | Minor errors | Major errors | Critical errors |
  |---|---|---:|---:|---:|
  | EN to PT-BR | Marian OPUS-MT | 47 | 17 | 0 |
  | EN to PT-BR | Tower+ 2B | 29 | 8 | 0 |
  | PT-BR to EN | Marian OPUS-MT | 32 | 12 | 0 |
  | PT-BR to EN | Tower+ 2B | 10 | 3 | 0 |

### Pairwise Stability


When the judge compared both translations directly, Tower+ was preferred approximately four out of five times:

- English to Portuguese: 79.4% Tower+ versus 20.6% Marian.
- Portuguese to English: 79.0% Tower+ versus 21.0% Marian.

| Direction | Stable comparisons | Unstable comparisons |
|---|---:|---:|
| EN to PT-BR | 34 | 2 |
| PT-BR to EN | 31 | 5 |

Only stable comparisons contributed to the preference rate.

With Tower+ selected as the initial translator, I could return to the original workflow question. Would the relevant commands run within the constraints of a public CI runner?

## Validating Locally Before Deployment

Before giving a GitHub Actions workflow permission to create branches, pull requests, or previews, I wanted to reproduce its computational steps locally. I therefore added a [Dockerized CI local test](https://github.com/abarbosa94/personal_blog/tree/main/experiments/scaling-my-posts/docker) to the experiment.

The idea is to approximate the resource envelope that GitHub Actions would provide. Therefore, the service runs Ubuntu with limits of four CPUs and 16 GB of memory, matching the documented resource envelope for a [standard public `ubuntu-latest` runner](https://docs.github.com/en/actions/reference/runners/github-hosted-runners). Inside that container, the rehearsal:

1. runs the focused experiment tests;
2. renders this notebook with Quarto;
3. optionally loads Tower+ and translates one short example in each direction; and
4. writes a JSON evidence report containing the input hash, stage results, runtime, visible resource limits, disk usage, and model revision.

The model cache lives in a named Docker volume, so subsequent runs do not need to download the model again.

I ran both local rehearsals successfully. The container reported a four-CPU cgroup quota and a 16 GB memory limit. The focused tests passed, and Quarto rendered the site in approximately 10 minutes. This is slow, but acceptable for this MVP. Moreover, the model smoke loaded the pinned 2.614-billion-parameter Tower+ revision and produced non-empty English-to-Portuguese and Portuguese-to-English translations. Its peak Python-process memory was approximately 5.7 GiB. A clean Tower+ cache occupied approximately 5.3 GB, and the CI image occupied approximately 0.65 GB, leaving room within the runner's 14 GB disk envelope for the repository and rendered preview. Docker Compose reports this footprint but does not enforce a disk quota, so the real runner remains the final check. The exact notebook hash, stage results, measurements, model revision, and limitations are recorded in a [sanitized public evidence summary](https://github.com/abarbosa94/personal_blog/blob/main/posts/data/translation-local-ci-evidence.json).

This creates an important evidence boundary. These results show that the tested translation, validation, and rendering commands work in a clean, resource-constrained Linux environment on my machine!


# Where the Bilingual Blog Goes Next

Under this dataset and judge, Tower+ 2B received better quality scores than Marian OPUS-MT in both directions. The direct pairwise evaluation preferred Tower+ approximately 79% of the time in both directions. The model is relatively heavy, but it ran on my benchmark CPU. The complete GitHub Actions workflow still needs to be implemented and tested. Here are some ideas I would like to explore next:

- Improve judge calibration: I noticed that the judge has a considerably wide confidence interval, partly because the pilot includes only 14 comparable items. As future work, I could review more cases to narrow the interval and sample ordinary posts and varied content types to broaden the evaluation dataset's coverage. If that is not enough, I could also refine the judge prompt—but that is another story.

- Reviewer agent: deploy a lightweight reviewer agent that can identify possible improvements to the translated prose when the translation PR is created.

- Faster models with better quality: the initial model may have 2 billion parameters and run on a CPU, but Marian models are faster and much lighter. As I write and evaluate more posts, I will gradually build a larger, better dataset. I could then fine-tune a Tower or Marian model for my needs or experiment with quantized models.


In this post, I explored how access to powerful coding agents through Codex reduced the cost of creating an evaluation dataset and obtaining useful measurements. The experiment also showed where human judgment remains essential when building an AI-powered feature that serves a real purpose. It is also important to account for the time and money spent on this work:

- Time: 2 days
- Money: USD 40 in prepaid credit (USD 20 for Codex and USD 20 for the Kimi judge). This is purchased credit, not the exact amount consumed by the experiment.


Until next time!

## References

::: {#refs}
:::